# GNN-CSP Ablation & Sensitivity Analysis

이 노트북은 GNN-CSP 모델의 성능을 분석하기 위한 실험을 수행합니다.

**주의: 아래의 '환경 설정 및 파이프라인 초기화' 셀을 반드시 가장 먼저 실행해주세요.**

In [ ]:
import gnn_csp_utils
import torch
import numpy as np
import os
from hiertable_rag.evaluation.metrics import calculate_teds
from tqdm.notebook import tqdm

# 1. 환경 설정 (프로젝트 경로 자동 추가)
gnn_csp_utils.setup_notebook_env()

# 2. 파이프라인 초기화
pipeline = gnn_csp_utils.get_pipeline()

print("\n--- GNN-CSP 분석 준비 완료 ---")

In [ ]:
def load_ground_truth(gt_dir: str, num_samples: int = 10) -> List[Dict[str, Any]]:
    """SciTSR Ground Truth 데이터를 로드합니다."""
    import json
    from typing import List, Dict, Any
    samples = []
    if not os.path.exists(gt_dir):
        return samples
        
    files = [f for f in os.listdir(gt_dir) if f.endswith('.json')]
    selected_files = files[:num_samples]
    
    for fname in selected_files:
        path = os.path.join(gt_dir, fname)
        with open(path, 'r') as f:
            data = json.load(f)
            if isinstance(data, list):
                cells_data = data
                gt_structure = {'cells': data}
            else:
                cells_data = data.get('cells', [])
                gt_structure = data
            
            samples.append({
                'filename': fname,
                'image': torch.zeros((3, 1000, 1000)),
                'ocr_boxes': [{'box': c.get('box', [0,0,0,0]), 'content': c.get('text', '')} for c in cells_data],
                'ground_truth': gt_structure
            })
    return samples

def run_experiment(pipeline, samples, config_name, use_csp, use_semantic):
    results = {'metrics': {'teds': [], 'time': [], 'row_acc': []}}
    for sample in tqdm(samples, desc=f"실험: {config_name}"):
        output = pipeline.parse(sample['image'], sample['ocr_boxes'], use_csp=use_csp, use_semantic=use_semantic)
        results['metrics']['teds'].append(calculate_teds(output['cells'], sample['ground_truth']))
        results['metrics']['time'].append(output['solve_time'])
        # 간단한 행 계산
        gt_rows = max([c['logical_coords'][2] for c in sample['ground_truth']['cells'] if 'logical_coords' in c]) + 1 if sample['ground_truth']['cells'] else 0
        results['metrics']['row_acc'].append(abs(output['num_rows'] - gt_rows))
    
    return {
        'mean_teds': np.mean(results['metrics']['teds']),
        'mean_time': np.mean(results['metrics']['time']),
        'mean_row_error': np.mean(results['metrics']['row_acc'])
    }

In [ ]:
GT_DIR = '/root/t1-9/data/external/scitsr/SciTsr_Logical/test/gt/'
samples = load_ground_truth(GT_DIR, 10)
print(f"데이터 로드 완료: {len(samples)}개 샘플")

## 실험 실행

In [ ]:
results = {}
for name, use_csp, use_sem in [('Full', True, True), ('No CSP', False, True), ('No Sem', True, False)]:
    results[name] = run_experiment(pipeline, samples, name, use_csp, use_sem)

print("\n--- 실험 결과 ---")
for name, res in results.items():
    print(f"{name:8s}: TEDS={res['mean_teds']:.4f}, 시간={res['mean_time']:.2f}s, Row에러={res['mean_row_error']:.2f}")